# Gold Recovery Prediction

Predicting recovery efficiency at two stages of an ore-processing workflow.

**Result:** The selected Random Forest produced test weighted sMAPE = 6.4367 versus 7.0453 for the constant baseline.

**Methods:** regression, industrial process data, custom metrics, cross-validation, Random Forest.

> This portfolio version removes course-review correspondence and repetitive instructional text. The analysis, models, and reported metrics are based on the original completed project. The source datasets are not included in this repository.


## 1. Setup and data quality checks

Training, test, and full process datasets are loaded. Recovery calculations, missing values, duplicates, and train/test feature differences are checked.


In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from IPython.display import display
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error, make_scorer

warnings.filterwarnings('ignore')


In [55]:
df_train = pd.read_csv(
    '/Users/kolotukhin.md/Downloads/jupyter_notebook/final_2/gold_recovery_train_new.csv'
)
df_test = pd.read_csv(
    '/Users/kolotukhin.md/Downloads/jupyter_notebook/final_2/gold_recovery_test_new.csv'
)
df_full = pd.read_csv(
    '/Users/kolotukhin.md/Downloads/jupyter_notebook/final_2/gold_recovery_full_new.csv'
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [56]:
df_list = [df_train, df_test, df_full]
for i in df_list:
    display(i.head())
    print(i.info())


,date,final.output.concentrate_ag,final.output.concentrate_pb,final.output.concentrate_sol,final.output.concentrate_au,final.output.recovery,final.output.tail_ag,final.output.tail_pb,final.output.tail_sol,final.output.tail_au,primary_cleaner.input.sulfate,primary_cleaner.input.depressant,primary_cleaner.input.feed_size,primary_cleaner.input.xanthate,primary_cleaner.output.concentrate_ag,primary_cleaner.output.concentrate_pb,primary_cleaner.output.concentrate_sol,primary_cleaner.output.concentrate_au,primary_cleaner.output.tail_ag,primary_cleaner.output.tail_pb,primary_cleaner.output.tail_sol,primary_cleaner.output.tail_au,primary_cleaner.state.floatbank8_a_air,primary_cleaner.state.floatbank8_a_level,primary_cleaner.state.floatbank8_b_air,primary_cleaner.state.floatbank8_b_level,primary_cleaner.state.floatbank8_c_air,primary_cleaner.state.floatbank8_c_level,primary_cleaner.state.floatbank8_d_air,primary_cleaner.state.floatbank8_d_level,rougher.calculation.sulfate_to_au_concentrate,rougher.calculation.floatbank10_sulfate_to_au_feed,rougher.calculation.floatbank11_sulfate_to_au_feed,rougher.calculation.au_pb_ratio,rougher.input.feed_ag,rougher.input.feed_pb,rougher.input.feed_rate,rougher.input.feed_size,rougher.input.feed_sol,rougher.input.feed_au,rougher.input.floatbank10_sulfate,rougher.input.floatbank10_xanthate,rougher.input.floatbank11_sulfate,rougher.input.floatbank11_xanthate,rougher.output.concentrate_ag,rougher.output.concentrate_pb,rougher.output.concentrate_sol,rougher.output.concentrate_au,rougher.output.recovery,rougher.output.tail_ag,rougher.output.tail_pb,rougher.output.tail_sol,rougher.output.tail_au,rougher.state.floatbank10_a_air,rougher.state.floatbank10_a_level,rougher.state.floatbank10_b_air,rougher.state.floatbank10_b_level,rougher.state.floatbank10_c_air,rougher.state.floatbank10_c_level,rougher.state.floatbank10_d_air,rougher.state.floatbank10_d_level,rougher.state.floatbank10_e_air,rougher.state.floatbank10_e_level,rougher.state.floatbank10_f_air,rougher.state.floatbank10_f_level,secondary_cleaner.output.tail_ag,secondary_cleaner.output.tail_pb,secondary_cleaner.output.tail_sol,secondary_cleaner.output.tail_au,secondary_cleaner.state.floatbank2_a_air,secondary_cleaner.state.floatbank2_a_level,secondary_cleaner.state.floatbank2_b_air,secondary_cleaner.state.floatbank2_b_level,secondary_cleaner.state.floatbank3_a_air,secondary_cleaner.state.floatbank3_a_level,secondary_cleaner.state.floatbank3_b_air,secondary_cleaner.state.floatbank3_b_level,secondary_cleaner.state.floatbank4_a_air,secondary_cleaner.state.floatbank4_a_level,secondary_cleaner.state.floatbank4_b_air,secondary_cleaner.state.floatbank4_b_level,secondary_cleaner.state.floatbank5_a_air,secondary_cleaner.state.floatbank5_a_level,secondary_cleaner.state.floatbank5_b_air,secondary_cleaner.state.floatbank5_b_level,secondary_cleaner.state.floatbank6_a_air,secondary_cleaner.state.floatbank6_a_level
0,2016-01-15 00:00:00,6.055403,9.889648,5.507324,42.192020,70.541216,10.411962,0.895447,16.904297,2.143149,127.092003,10.128295,7.25,0.988759,8.547551,10.389648,19.529297,34.174427,14.936526,2.534912,7.476074,2.106679,1549.775757,-498.912140,1551.434204,-516.403442,1549.873901,-498.666595,1554.367432,-493.428131,41885.707031,3481.779053,3520.337158,2.838687,6.100378,2.284912,523.546326,55.486599,36.808594,6.486150,11.986616,6.007990,11.836743,6.005818,11.500771,7.101074,28.029297,19.793808,87.107763,5.008018,0.508728,19.154297,1.170244,999.706909,-404.066986,1603.011353,-434.715027,1602.375000,-442.204468,1598.937256,-451.294128,1404.472046,-455.462982,1416.354980,-451.939636,14.500184,4.694824,8.764648,2.606185,25.853109,-498.526489,23.893660,-501.406281,23.961798,-495.262817,21.940409,-499.340973,14.016835,-502.488007,12.099931,-504.715942,9.925633,-498.310211,8.079666,-500.470978,14.151341,-605.841980
1,2016-01-15 01:00:00,6.029369,9.968944,5.257781,42.701629,69.266198,10.462676,0.927452,16.634514,2.224930,125.629232,10.296251,7.25,1.002663,8.558743,10.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14149 entries, 0 to 14148
Data columns (total 87 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   date                                                14149 non-null  object 
 1   final.output.concentrate_ag                         14148 non-null  float64
 2   final.output.concentrate_pb                         14148 non-null  float64
 3   final.output.concentrate_sol                        13938 non-null  float64
 4   final.output.concentrate_au                         14149 non-null  float64
 5   final.output.recovery                               14149 non-null  float64
 6   final.output.tail_ag                                14149 non-null  float64
 7   final.output.tail_pb                                14049 non-null  float64
 8   final.output.tail_sol                               14144 non-null  float64


,date,primary_cleaner.input.sulfate,primary_cleaner.input.depressant,primary_cleaner.input.feed_size,primary_cleaner.input.xanthate,primary_cleaner.state.floatbank8_a_air,primary_cleaner.state.floatbank8_a_level,primary_cleaner.state.floatbank8_b_air,primary_cleaner.state.floatbank8_b_level,primary_cleaner.state.floatbank8_c_air,primary_cleaner.state.floatbank8_c_level,primary_cleaner.state.floatbank8_d_air,primary_cleaner.state.floatbank8_d_level,rougher.input.feed_ag,rougher.input.feed_pb,rougher.input.feed_rate,rougher.input.feed_size,rougher.input.feed_sol,rougher.input.feed_au,rougher.input.floatbank10_sulfate,rougher.input.floatbank10_xanthate,rougher.input.floatbank11_sulfate,rougher.input.floatbank11_xanthate,rougher.state.floatbank10_a_air,rougher.state.floatbank10_a_level,rougher.state.floatbank10_b_air,rougher.state.floatbank10_b_level,rougher.state.floatbank10_c_air,rougher.state.floatbank10_c_level,rougher.state.floatbank10_d_air,rougher.state.floatbank10_d_level,rougher.state.floatbank10_e_air,rougher.state.floatbank10_e_level,rougher.state.floatbank10_f_air,rougher.state.floatbank10_f_level,secondary_cleaner.state.floatbank2_a_air,secondary_cleaner.state.floatbank2_a_level,secondary_cleaner.state.floatbank2_b_air,secondary_cleaner.state.floatbank2_b_level,secondary_cleaner.state.floatbank3_a_air,secondary_cleaner.state.floatbank3_a_level,secondary_cleaner.state.floatbank3_b_air,secondary_cleaner.state.floatbank3_b_level,secondary_cleaner.state.floatbank4_a_air,secondary_cleaner.state.floatbank4_a_level,secondary_cleaner.state.floatbank4_b_air,secondary_cleaner.state.floatbank4_b_level,secondary_cleaner.state.floatbank5_a_air,secondary_cleaner.state.floatbank5_a_level,secondary_cleaner.state.floatbank5_b_air,secondary_cleaner.state.floatbank5_b_level,secondary_cleaner.state.floatbank6_a_air,secondary_cleaner.state.floatbank6_a_level
0,2016-09-01 00:59:59,210.800909,14.993118,8.080000,1.005021,1398.981301,-500.225577,1399.144926,-499.919735,1400.102998,-500.704369,1399.001695,-499.485144,13.128569,5.636710,489.793655,62.710452,42.022440,12.084103,16.922877,6.153479,16.867518,6.150709,1001.849317,-350.301265,1249.738612,-399.107531,1249.750520,-399.396730,1198.287855,-399.488594,999.472250,-399.530962,949.566189,-398.180807,24.938200,-500.490963,14.947607,-500.013837,20.018166,-450.395995,13.987058,-449.831513,12.023554,-497.795834,8.016656,-501.289139,7.946562,-432.317850,4.872511,-500.037437,26.705889,-499.709414
1,2016-09-01 01:59:59,215.392455,14.987471,8.080000,0.990469,1398.777912,-500.057435,1398.055362,-499.778182,1396.151033,-499.240168,1399.508091,-500.416324,13.036280,5.526065,490.104534,61.961163,41.187903,11.918977,17.002532,5.999043,16.996063,6.002315,998.690785,-350.425704,1248.395457,-399.946025,1249.514451,-399.630810,1200.506459,-399.942434,1000.002114,-399.491528,950.200009,-405.787514,24.923466,-499.813963,14.930873,-500.764452,19.988503,-450.114024,14.092981,-450.059163,12.058140,-498.695773,8.130979,-499.634209,7.958270,-525.839648,4.878850,-500.162375,25.019940,-499.819438
2,2016-09-01 02:59:59,215.259946,12.884934,7.786667,0.996043,1398.493666,-500.868360,1398.860436,-499.764529,1398.075709,-502.151509,1399.497088,-499.715479,13.138412,5.426561,489.617969,66.903807,42.546294,12.091332,16.992520,5.850632,16.982096,5.853594,998.516720,-349.783538,1247.440748,-400.264330,1248.207204,-401.074405,1199.770026,-400.790582,999.925519,-399.236999,950.320481,-400.864340,24.908732,-500.304035,14.996511,-500.993821,20.039689,-450.263164,14.077881,-449.660997,11.962366,-498.767484,8.096893,-500.827423,8.071056,-500.801673,4.905125,-499.828510,24.994862,-500.622559
3,2016-09-01 03:59:59,215.336236,12.006805,7.640000,0.863514,1399.618111,-498.863574,1397.440120,-499.211024,1400.129303,-498.355873,1401.065165,-501.038738,12.400507,5.114758,476.618130,59.865919,41.060195,12.181597,16.532144,5.801935,16.515277,5.803657,1000.276604,-350.172597,1251.322675,-398.655289,1250.493702,-399.746728,1199.398691,-397.50

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5290 entries, 0 to 5289
Data columns (total 53 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   date                                        5290 non-null   object 
 1   primary_cleaner.input.sulfate               5286 non-null   float64
 2   primary_cleaner.input.depressant            5285 non-null   float64
 3   primary_cleaner.input.feed_size             5290 non-null   float64
 4   primary_cleaner.input.xanthate              5286 non-null   float64
 5   primary_cleaner.state.floatbank8_a_air      5290 non-null   float64
 6   primary_cleaner.state.floatbank8_a_level    5290 non-null   float64
 7   primary_cleaner.state.floatbank8_b_air      5290 non-null   float64
 8   primary_cleaner.state.floatbank8_b_level    5290 non-null   float64
 9   primary_cleaner.state.floatbank8_c_air      5290 non-null   float64
 10  primary_clea

,date,final.output.concentrate_ag,final.output.concentrate_pb,final.output.concentrate_sol,final.output.concentrate_au,final.output.recovery,final.output.tail_ag,final.output.tail_pb,final.output.tail_sol,final.output.tail_au,primary_cleaner.input.sulfate,primary_cleaner.input.depressant,primary_cleaner.input.feed_size,primary_cleaner.input.xanthate,primary_cleaner.output.concentrate_ag,primary_cleaner.output.concentrate_pb,primary_cleaner.output.concentrate_sol,primary_cleaner.output.concentrate_au,primary_cleaner.output.tail_ag,primary_cleaner.output.tail_pb,primary_cleaner.output.tail_sol,primary_cleaner.output.tail_au,primary_cleaner.state.floatbank8_a_air,primary_cleaner.state.floatbank8_a_level,primary_cleaner.state.floatbank8_b_air,primary_cleaner.state.floatbank8_b_level,primary_cleaner.state.floatbank8_c_air,primary_cleaner.state.floatbank8_c_level,primary_cleaner.state.floatbank8_d_air,primary_cleaner.state.floatbank8_d_level,rougher.calculation.sulfate_to_au_concentrate,rougher.calculation.floatbank10_sulfate_to_au_feed,rougher.calculation.floatbank11_sulfate_to_au_feed,rougher.calculation.au_pb_ratio,rougher.input.feed_ag,rougher.input.feed_pb,rougher.input.feed_rate,rougher.input.feed_size,rougher.input.feed_sol,rougher.input.feed_au,rougher.input.floatbank10_sulfate,rougher.input.floatbank10_xanthate,rougher.input.floatbank11_sulfate,rougher.input.floatbank11_xanthate,rougher.output.concentrate_ag,rougher.output.concentrate_pb,rougher.output.concentrate_sol,rougher.output.concentrate_au,rougher.output.recovery,rougher.output.tail_ag,rougher.output.tail_pb,rougher.output.tail_sol,rougher.output.tail_au,rougher.state.floatbank10_a_air,rougher.state.floatbank10_a_level,rougher.state.floatbank10_b_air,rougher.state.floatbank10_b_level,rougher.state.floatbank10_c_air,rougher.state.floatbank10_c_level,rougher.state.floatbank10_d_air,rougher.state.floatbank10_d_level,rougher.state.floatbank10_e_air,rougher.state.floatbank10_e_level,rougher.state.floatbank10_f_air,rougher.state.floatbank10_f_level,secondary_cleaner.output.tail_ag,secondary_cleaner.output.tail_pb,secondary_cleaner.output.tail_sol,secondary_cleaner.output.tail_au,secondary_cleaner.state.floatbank2_a_air,secondary_cleaner.state.floatbank2_a_level,secondary_cleaner.state.floatbank2_b_air,secondary_cleaner.state.floatbank2_b_level,secondary_cleaner.state.floatbank3_a_air,secondary_cleaner.state.floatbank3_a_level,secondary_cleaner.state.floatbank3_b_air,secondary_cleaner.state.floatbank3_b_level,secondary_cleaner.state.floatbank4_a_air,secondary_cleaner.state.floatbank4_a_level,secondary_cleaner.state.floatbank4_b_air,secondary_cleaner.state.floatbank4_b_level,secondary_cleaner.state.floatbank5_a_air,secondary_cleaner.state.floatbank5_a_level,secondary_cleaner.state.floatbank5_b_air,secondary_cleaner.state.floatbank5_b_level,secondary_cleaner.state.floatbank6_a_air,secondary_cleaner.state.floatbank6_a_level
0,2016-01-15 00:00:00,6.055403,9.889648,5.507324,42.192020,70.541216,10.411962,0.895447,16.904297,2.143149,127.092003,10.128295,7.25,0.988759,8.547551,10.389648,19.529297,34.174427,14.936526,2.534912,7.476074,2.106679,1549.775757,-498.912140,1551.434204,-516.403442,1549.873901,-498.666595,1554.367432,-493.428131,41885.707031,3481.779053,3520.337158,2.838687,6.100378,2.284912,523.546326,55.486599,36.808594,6.486150,11.986616,6.007990,11.836743,6.005818,11.500771,7.101074,28.029297,19.793808,87.107763,5.008018,0.508728,19.154297,1.170244,999.706909,-404.066986,1603.011353,-434.715027,1602.375000,-442.204468,1598.937256,-451.294128,1404.472046,-455.462982,1416.354980,-451.939636,14.500184,4.694824,8.764648,2.606185,25.853109,-498.526489,23.893660,-501.406281,23.961798,-495.262817,21.940409,-499.340973,14.016835,-502.488007,12.099931,-504.715942,9.925633,-498.310211,8.079666,-500.470978,14.151341,-605.841980
1,2016-01-15 01:00:00,6.029369,9.968944,5.257781,42.701629,69.266198,10.462676,0.927452,16.634514,2.224930,125.629232,10.296251,7.25,1.002663,8.558743,10.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19439 entries, 0 to 19438
Data columns (total 87 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   date                                                19439 non-null  object 
 1   final.output.concentrate_ag                         19438 non-null  float64
 2   final.output.concentrate_pb                         19438 non-null  float64
 3   final.output.concentrate_sol                        19228 non-null  float64
 4   final.output.concentrate_au                         19439 non-null  float64
 5   final.output.recovery                               19439 non-null  float64
 6   final.output.tail_ag                                19438 non-null  float64
 7   final.output.tail_pb                                19338 non-null  float64
 8   final.output.tail_sol                               19433 non-null  float64


In [57]:
t = df_train['rougher.output.tail_au']
f = df_train['rougher.input.feed_au']
c = df_train['rougher.output.concentrate_au']

r_o_recovery_calc = ((c * (f - t)) / (f * (c - t))) * 100

r_o_recovery_data = df_train['rougher.output.recovery']

print('MAE =', round(mean_absolute_error(r_o_recovery_data, r_o_recovery_calc), 10))


MAE = 0.0


In [58]:
set(df_train.columns) - set(df_test.columns)


{'final.output.concentrate_ag',
 'final.output.concentrate_au',
 'final.output.concentrate_pb',
 'final.output.concentrate_sol',
 'final.output.recovery',
 'final.output.tail_ag',
 'final.output.tail_au',
 'final.output.tail_pb',
 'final.output.tail_sol',
 'primary_cleaner.output.concentrate_ag',
 'primary_cleaner.output.concentrate_au',
 'primary_cleaner.output.concentrate_pb',
 'primary_cleaner.output.concentrate_sol',
 'primary_cleaner.output.tail_ag',
 'primary_cleaner.output.tail_au',
 'primary_cleaner.output.tail_pb',
 'primary_cleaner.output.tail_sol',
 'rougher.calculation.au_pb_ratio',
 'rougher.calculation.floatbank10_sulfate_to_au_feed',
 'rougher.calculation.floatbank11_sulfate_to_au_feed',
 'rougher.calculation.sulfate_to_au_concentrate',
 'rougher.output.concentrate_ag',
 'rougher.output.concentrate_au',
 'rougher.output.concentrate_pb',
 'rougher.output.concentrate_sol',
 'rougher.output.recovery',
 'rougher.output.tail_ag',
 'rougher.output.tail_au',
 'rougher.output.ta

In [59]:
print(set(df_train.columns) - set(df_test.columns))


{'secondary_cleaner.output.tail_pb', 'rougher.output.tail_au', 'primary_cleaner.output.concentrate_sol', 'primary_cleaner.output.tail_au', 'final.output.concentrate_ag', 'primary_cleaner.output.concentrate_pb', 'primary_cleaner.output.tail_sol', 'rougher.output.concentrate_au', 'secondary_cleaner.output.tail_sol', 'primary_cleaner.output.concentrate_ag', 'rougher.output.concentrate_sol', 'rougher.output.tail_sol', 'rougher.output.concentrate_pb', 'final.output.concentrate_pb', 'rougher.calculation.sulfate_to_au_concentrate', 'primary_cleaner.output.tail_ag', 'primary_cleaner.output.tail_pb', 'secondary_cleaner.output.tail_ag', 'rougher.output.tail_ag', 'final.output.tail_sol', 'final.output.tail_au', 'rougher.output.concentrate_ag', 'rougher.calculation.floatbank11_sulfate_to_au_feed', 'rougher.output.tail_pb', 'final.output.tail_ag', 'rougher.calculation.floatbank10_sulfate_to_au_feed', 'final.output.recovery', 'rougher.calculation.au_pb_ratio', 'secondary_cleaner.output.tail_au', 'pr

In [60]:

pd.DataFrame(df_train.isna().mean()
             .to_frame(name='Missing share')
             .query('Missing share > 0')['Missing share'])\
    .style.background_gradient('coolwarm')\
    .format({'Missing share': '{:.2%}'})


In [61]:

pd.DataFrame(df_test.isna().mean()
             .to_frame(name='Missing share')
             .query('Missing share > 0')['Missing share'])\
    .style.background_gradient('coolwarm')\
    .format({'Missing share': '{:.2%}'})


In [62]:

pd.DataFrame(df_full.isna().mean()
             .to_frame(name='Missing share')
             .query('Missing share > 0')['Missing share'])\
    .style.background_gradient('coolwarm')\
    .format({'Missing share': '{:.2%}'})


In [63]:

df_train = df_train.fillna(method='bfill')
df_test = df_test.fillna(method='bfill')
df_full = df_full.fillna(method='bfill')


In [64]:

pd.DataFrame(df_train.isna().mean()
             .to_frame(name='Missing share')
             .query('Missing share > 0')['Missing share'])\
    .style.background_gradient('coolwarm')\
    .format({'Missing share': '{:.2%}'})


In [65]:

pd.DataFrame(df_test.isna().mean()
             .to_frame(name='Missing share')
             .query('Missing share > 0')['Missing share'])\
    .style.background_gradient('coolwarm')\
    .format({'Missing share': '{:.2%}'})


In [66]:

pd.DataFrame(df_full.isna().mean()
             .to_frame(name='Missing share')
             .query('Missing share > 0')['Missing share'])\
    .style.background_gradient('coolwarm')\
    .format({'Missing share': '{:.2%}'})


In [67]:
df_list = [df_train, df_test, df_full]
for i in df_list:
    i.isna().sum()
    print(i.isna().sum())


date                                                  0
final.output.concentrate_ag                           0
final.output.concentrate_pb                           0
final.output.concentrate_sol                          0
final.output.concentrate_au                           0
final.output.recovery                                 0
final.output.tail_ag                                  0
final.output.tail_pb                                  0
final.output.tail_sol                                 0
final.output.tail_au                                  0
primary_cleaner.input.sulfate                         0
primary_cleaner.input.depressant                      0
primary_cleaner.input.feed_size                       0
primary_cleaner.input.xanthate                        0
primary_cleaner.output.concentrate_ag                 0
primary_cleaner.output.concentrate_pb                 0
primary_cleaner.output.concentrate_sol                0
primary_cleaner.output.concentrate_au           

In [68]:

for i in df_list:
    print(i.duplicated().value_counts())
    print(i.duplicated().sum())


False    14149
dtype: int64
0
False    5290
dtype: int64
0
False    19439
dtype: int64
0


## 2. Process analysis

Metal concentrations, feed-particle distributions, and total concentration at successive processing stages are examined.


In [69]:
plt.figure(figsize=(30, 20))

df_full['rougher.input.feed_au'].hist(
    bins=80, label='rougher.input.feed_au', alpha=0.4, density=True)
df_full['rougher.output.concentrate_au'].hist(
    bins=80, label='rougher.output.concentrate_au', alpha=0.4, density=True)
df_full['primary_cleaner.output.concentrate_au'].hist(
    bins=80, label='primary_cleaner.output.concentrate_au', alpha=0.4, density=True)
df_full['final.output.concentrate_au'].hist(
    bins=80, label='final.output.concentrate_au', alpha=0.4, density=True)

plt.title(
    'Gold (Au) concentration across processing stages', fontsize=30)
plt.legend(loc='upper right', prop={'size': 25})
plt.xlabel('Value', fontsize=30)
plt.ylabel('Count', fontsize=30)
plt.show()


<Figure size 2160x1440 with 1 Axes>

In [70]:
df_full[['rougher.input.feed_au',
         'rougher.output.concentrate_au',
         'primary_cleaner.output.concentrate_au',
         'final.output.concentrate_au']].mean()


rougher.input.feed_au                     8.266097
rougher.output.concentrate_au            19.772000
primary_cleaner.output.concentrate_au    32.119400
final.output.concentrate_au              44.076513
dtype: float64

In [71]:
plt.figure(figsize=(30, 20))

df_full['rougher.input.feed_ag'].hist(
    bins=80, label='rougher.input.feed_ag', alpha=0.4, density=True)
df_full['rougher.output.concentrate_ag'].hist(
    bins=80, label='rougher.output.concentrate_ag', alpha=0.4, density=True)
df_full['primary_cleaner.output.concentrate_ag'].hist(
    bins=80, label='primary_cleaner.output.concentrate_ag', alpha=0.4, density=True)
df_full['final.output.concentrate_ag'].hist(
    bins=80, label='final.output.concentrate_ag', alpha=0.4, density=True)

plt.title(
    'Silver (Ag) concentration across processing stages', fontsize=30)
plt.legend(loc='upper right', prop={'size': 25})
plt.xlabel('Value', fontsize=30)
plt.ylabel('Count', fontsize=30)
plt.show()


<Figure size 2160x1440 with 1 Axes>

In [72]:
df_full[['rougher.input.feed_ag',
         'rougher.output.concentrate_ag',
         'primary_cleaner.output.concentrate_ag',
         'final.output.concentrate_ag']].mean()


rougher.input.feed_ag                     8.794927
rougher.output.concentrate_ag            11.994759
primary_cleaner.output.concentrate_ag     8.442408
final.output.concentrate_ag               5.168382
dtype: float64

In [73]:
plt.figure(figsize=(30, 20))

df_full['rougher.input.feed_pb'].hist(
    bins=80, label='rougher.input.feed_pb', alpha=0.4, density=True)
df_full['rougher.output.concentrate_pb'].hist(
    bins=80, label='rougher.output.concentrate_pb', alpha=0.4, density=True)
df_full['primary_cleaner.output.concentrate_pb'].hist(
    bins=80, label='primary_cleaner.output.concentrate_pb', alpha=0.4, density=True)
df_full['final.output.concentrate_pb'].hist(
    bins=80, label='final.output.concentrate_pb', alpha=0.4, density=True)

plt.title(
    'Lead (Pb) concentration across processing stages', fontsize=30)
plt.legend(loc='upper right', prop={'size': 25})
plt.xlabel('Value', fontsize=30)
plt.ylabel('Count', fontsize=30)
plt.show()


<Figure size 2160x1440 with 1 Axes>

In [74]:
df_full[['rougher.input.feed_pb',
         'rougher.output.concentrate_pb',
         'primary_cleaner.output.concentrate_pb',
         'final.output.concentrate_pb']].mean()


rougher.input.feed_pb                    3.585850
rougher.output.concentrate_pb            7.606272
primary_cleaner.output.concentrate_pb    9.810430
final.output.concentrate_pb              9.978817
dtype: float64

In [75]:
plt.figure(figsize=(20, 10))

df_train['rougher.input.feed_size'].hist(
    bins=80, label='df_train-rougher.input.feed_size', alpha=0.4, density=True)

df_test['rougher.input.feed_size'].hist(
    bins=80, label='df_test-rougher.input.feed_size', alpha=0.4, density=True)

plt.title('Feed particle-size distribution in training and test data', fontsize=30)
plt.legend(loc='upper right', prop={'size': 25})
plt.xlabel('Value', fontsize=30)
plt.ylabel('Count', fontsize=30)
plt.show()


<Figure size 1440x720 with 1 Axes>

In [76]:
plt.figure(figsize=(20, 10))

df_train['primary_cleaner.input.feed_size'].hist(
    bins=80, label='df_train-primary_cleaner.input.feed_size', alpha=0.4, density=True)

df_test['primary_cleaner.input.feed_size'].hist(
    bins=80, label='df_test-primary_cleaner.input.feed_size', alpha=0.4, density=True)

plt.title('Feed particle-size distribution in training and test data', fontsize=30)
plt.legend(loc='upper right', prop={'size': 25})
plt.xlabel('Value', fontsize=30)
plt.ylabel('Count', fontsize=30)
plt.show()


<Figure size 1440x720 with 1 Axes>

In [77]:
df_full['rougher.input.feed_together'] = df_full[['rougher.input.feed_au',
                                                 'rougher.input.feed_ag',
                                                  'rougher.input.feed_pb']].sum(axis=1)

df_full['rougher.output.concentrate_together'] = df_full[['rougher.output.concentrate_au',
                                                         'rougher.output.concentrate_ag',
                                                          'rougher.output.concentrate_pb']].sum(axis=1)

df_full['primary_cleaner.output.concentrate_together'] = df_full[['primary_cleaner.output.concentrate_au',
                                                                 'primary_cleaner.output.concentrate_ag',
                                                                  'primary_cleaner.output.concentrate_pb']].sum(axis=1)

df_full['final.output.concentrate_together'] = df_full[['final.output.concentrate_au',
                                                       'final.output.concentrate_ag',
                                                        'final.output.concentrate_pb']].sum(axis=1)


In [78]:
plt.figure(figsize=(20, 10))

df_full['rougher.input.feed_together'].hist(
    bins=80, label='rougher.input.feed_together', alpha=0.4, density=True)

df_full['rougher.output.concentrate_together'].hist(
    bins=80, label='rougher.output.concentrate_together', alpha=0.4, density=True)

df_full['primary_cleaner.output.concentrate_together'].hist(
    bins=80, label='primary_cleaner.output.concentrate_together', alpha=0.4, density=True)

df_full['final.output.concentrate_together'].hist(
    bins=80, label='final.output.concentrate_together', alpha=0.4, density=True)

plt.title('Total concentration across processing stages', fontsize=30)
plt.legend(loc='upper left', prop={'size': 25})
plt.xlabel('Value', fontsize=30)
plt.ylabel('Count', fontsize=30)
plt.show()


<Figure size 1440x720 with 1 Axes>

In [79]:
df_full[['rougher.input.feed_together',
         'rougher.output.concentrate_together',
         'primary_cleaner.output.concentrate_together',
         'final.output.concentrate_together']].boxplot(figsize=(20, 10))
plt.title(
    'Total concentration by processing stage', fontsize=25)
plt.xlabel('Total concentration by stage', fontsize=30)
plt.ylabel('Value', fontsize=30)
plt.show()


<Figure size 1440x720 with 1 Axes>

In [80]:

final_q1 = np.quantile(
    df_full['rougher.input.feed_together'], 0.25)
final_q3 = np.quantile(
    df_full['rougher.input.feed_together'], 0.75)
delta_q3_q1 = final_q3 - final_q1
rougher_input_moustache = final_q1 - 1.5 * delta_q3_q1
print(' final_q1: ', final_q1, '\n', 'final_q3: ', final_q3, '\n', 'delta_q3_q1: ',
      delta_q3_q1, '\n', 'rougher_left_moustache: ', rougher_input_moustache)

final_q1 = np.quantile(
    df_full['rougher.output.concentrate_together'], 0.25)
final_q3 = np.quantile(
    df_full['rougher.output.concentrate_together'], 0.75)
delta_q3_q1 = final_q3 - final_q1
rougher_output_moustache = final_q1 - 1.5 * delta_q3_q1
print(' final_q1: ', final_q1, '\n', 'final_q3: ', final_q3, '\n', 'delta_q3_q1: ',
      delta_q3_q1, '\n', 'rougher_left_moustache: ', rougher_output_moustache)

final_q1 = np.quantile(
    df_full['primary_cleaner.output.concentrate_together'], 0.25)
final_q3 = np.quantile(
    df_full['primary_cleaner.output.concentrate_together'], 0.75)
delta_q3_q1 = final_q3 - final_q1
primary_cleaner_moustache = final_q1 - 1.5 * delta_q3_q1
print(' final_q1: ', final_q1, '\n', 'final_q3: ', final_q3, '\n', 'delta_q3_q1: ',
      delta_q3_q1, '\n', 'rougher_left_moustache: ', primary_cleaner_moustache)

final_q1 = np.quantile(
    df_full['final.output.concentrate_together'], 0.25)
final_q3 = np.quantile(
    df_full['final.output.concentrate_together'], 0.75)
delta_q3_q1 = final_q3 - final_q1
final_output_moustache = final_q1 - 1.5 * delta_q3_q1
print(' final_q1: ', final_q1, '\n', 'final_q3: ', final_q3, '\n', 'delta_q3_q1: ',
      delta_q3_q1, '\n', 'rougher_left_moustache: ', final_output_moustache)

df_full = \
    df_full[(df_full['rougher.input.feed_together'] >= rougher_input_moustache)
            & (df_full['rougher.output.concentrate_together'] >= rougher_output_moustache)
            & (df_full['primary_cleaner.output.concentrate_together'] >= primary_cleaner_moustache)
            & (df_full['final.output.concentrate_together'] >= final_output_moustache)]
df_full.shape


 final_q1:  17.252879439403607 
 final_q3:  23.79738079415382 
 delta_q3_q1:  6.5445013547502136 
 rougher_left_moustache:  7.436127407278287
 final_q1:  38.39463885885074 
 final_q3:  42.36510159612476 
 delta_q3_q1:  3.970462737274019 
 rougher_left_moustache:  32.43894475293971
 final_q1:  50.05429015332034 
 final_q3:  53.38689803478148 
 delta_q3_q1:  3.3326078814611435 
 rougher_left_moustache:  45.05537833112862
 final_q1:  59.29832252160742 
 final_q3:  61.09666661301081 
 delta_q3_q1:  1.7983440914033935 
 rougher_left_moustache:  56.60080638450233


(16824, 91)

In [81]:

df_train['rougher.input.feed_together'] = df_train[['rougher.input.feed_au',
                                                    'rougher.input.feed_ag',
                                                    'rougher.input.feed_pb']].sum(axis=1)

df_train['rougher.output.concentrate_together'] = df_train[['rougher.output.concentrate_au',
                                                            'rougher.output.concentrate_ag',
                                                            'rougher.output.concentrate_pb']].sum(axis=1)

df_train['primary_cleaner.output.concentrate_together'] = df_train[['primary_cleaner.output.concentrate_au',
                                                                    'primary_cleaner.output.concentrate_ag',
                                                                    'primary_cleaner.output.concentrate_pb']].sum(axis=1)

df_train['final.output.concentrate_together'] = df_train[['final.output.concentrate_au',
                                                         'final.output.concentrate_ag',
                                                          'final.output.concentrate_pb']].sum(axis=1)

final_q1 = np.quantile(
    df_train['rougher.input.feed_together'], 0.25)
final_q3 = np.quantile(
    df_train['rougher.input.feed_together'], 0.75)
delta_q3_q1 = final_q3 - final_q1
rougher_input_moustache = final_q1 - 1.5 * delta_q3_q1
print(' final_q1: ', final_q1, '\n', 'final_q3: ', final_q3, '\n', 'delta_q3_q1: ',
      delta_q3_q1, '\n', 'rougher_left_moustache: ', rougher_input_moustache)

final_q1 = np.quantile(
    df_train['rougher.output.concentrate_together'], 0.25)
final_q3 = np.quantile(
    df_train['rougher.output.concentrate_together'], 0.75)
delta_q3_q1 = final_q3 - final_q1
rougher_output_moustache = final_q1 - 1.5 * delta_q3_q1
print(' final_q1: ', final_q1, '\n', 'final_q3: ', final_q3, '\n', 'delta_q3_q1: ',
      delta_q3_q1, '\n', 'rougher_left_moustache: ', rougher_output_moustache)

final_q1 = np.quantile(
    df_train['primary_cleaner.output.concentrate_together'], 0.25)
final_q3 = np.quantile(
    df_train['primary_cleaner.output.concentrate_together'], 0.75)
delta_q3_q1 = final_q3 - final_q1
primary_cleaner_moustache = final_q1 - 1.5 * delta_q3_q1
print(' final_q1: ', final_q1, '\n', 'final_q3: ', final_q3, '\n', 'delta_q3_q1: ',
      delta_q3_q1, '\n', 'rougher_left_moustache: ', primary_cleaner_moustache)

final_q1 = np.quantile(
    df_train['final.output.concentrate_together'], 0.25)
final_q3 = np.quantile(
    df_train['final.output.concentrate_together'], 0.75)
delta_q3_q1 = final_q3 - final_q1
final_output_moustache = final_q1 - 1.5 * delta_q3_q1
print(' final_q1: ', final_q1, '\n', 'final_q3: ', final_q3, '\n', 'delta_q3_q1: ',
      delta_q3_q1, '\n', 'rougher_left_moustache: ', final_output_moustache)

df_train = \
    df_train[(df_train['rougher.input.feed_together'] >= rougher_input_moustache)
             & (df_train['rougher.output.concentrate_together'] >= rougher_output_moustache)
             & (df_train['primary_cleaner.output.concentrate_together'] >= primary_cleaner_moustache)
             & (df_train['final.output.concentrate_together'] >= final_output_moustache)]
df_train.shape


 final_q1:  16.85094389992646 
 final_q3:  22.571604760055195 
 delta_q3_q1:  5.720660860128735 
 rougher_left_moustache:  8.269952609733357
 final_q1:  37.92562121680521 
 final_q3:  41.79680529926956 
 delta_q3_q1:  3.871184082464346 
 rougher_left_moustache:  32.11884509310869
 final_q1:  49.63263744789611 
 final_q3:  53.44039223713346 
 delta_q3_q1:  3.8077547892373502 
 rougher_left_moustache:  43.92100526404008
 final_q1:  59.201504698939786 
 final_q3:  61.05482903481173 
 delta_q3_q1:  1.853324335871946 
 rougher_left_moustache:  56.42151819513187


(12301, 91)

In [82]:
# df_test.info()


In [83]:

df_test['rougher.input.feed_together'] = df_test[['rougher.input.feed_au',
                                                  'rougher.input.feed_ag',
                                                  'rougher.input.feed_pb']].sum(axis=1)

final_q1 = np.quantile(
    df_test['rougher.input.feed_together'], 0.25)
final_q3 = np.quantile(
    df_test['rougher.input.feed_together'], 0.75)
delta_q3_q1 = final_q3 - final_q1
rougher_input_moustache = final_q1 - 1.5 * delta_q3_q1
print(' final_q1: ', final_q1, '\n', 'final_q3: ', final_q3, '\n', 'delta_q3_q1: ',
      delta_q3_q1, '\n', 'rougher_left_moustache: ', rougher_input_moustache)

df_test = df_test[(df_test['rougher.input.feed_together']
                   >= rougher_input_moustache)]
df_test.shape


 final_q1:  19.81761236061428 
 final_q3:  25.114266429666557 
 delta_q3_q1:  5.296654069052277 
 rougher_left_moustache:  11.872631257035865


(5258, 54)

## 3. Modelling

Separate targets are prepared for rougher and final recovery. Models are compared with a custom weighted sMAPE scorer.


In [84]:
df_test = df_test.drop(['rougher.input.feed_together'], axis=1)


In [85]:
df_test_target = df_test.merge(
    df_full.loc[:, ['date', 'rougher.output.recovery', 'final.output.recovery']])


In [86]:
target_rougher_test = df_test_target['rougher.output.recovery']

target_final_test = df_test_target['final.output.recovery']


In [87]:
df_test_target = df_test_target.drop(['date'], axis=1)
df_test_target = df_test_target.drop(['rougher.output.recovery'], axis=1)
df_test_target = df_test_target.drop(['final.output.recovery'], axis=1)
df_train = df_train.drop(['date'], axis=1)


In [88]:
features_rougher_test = df_test_target
features_final_test = df_test_target


In [89]:
target_rougher_train = df_train['rougher.output.recovery']
target_final_train = df_train['final.output.recovery']


In [90]:
data_delete = set(df_train) - set(df_test)
display(data_delete)
df_train = df_train.drop(data_delete, axis='columns')


{'final.output.concentrate_ag',
 'final.output.concentrate_au',
 'final.output.concentrate_pb',
 'final.output.concentrate_sol',
 'final.output.concentrate_together',
 'final.output.recovery',
 'final.output.tail_ag',
 'final.output.tail_au',
 'final.output.tail_pb',
 'final.output.tail_sol',
 'primary_cleaner.output.concentrate_ag',
 'primary_cleaner.output.concentrate_au',
 'primary_cleaner.output.concentrate_pb',
 'primary_cleaner.output.concentrate_sol',
 'primary_cleaner.output.concentrate_together',
 'primary_cleaner.output.tail_ag',
 'primary_cleaner.output.tail_au',
 'primary_cleaner.output.tail_pb',
 'primary_cleaner.output.tail_sol',
 'rougher.calculation.au_pb_ratio',
 'rougher.calculation.floatbank10_sulfate_to_au_feed',
 'rougher.calculation.floatbank11_sulfate_to_au_feed',
 'rougher.calculation.sulfate_to_au_concentrate',
 'rougher.input.feed_together',
 'rougher.output.concentrate_ag',
 'rougher.output.concentrate_au',
 'rougher.output.concentrate_pb',
 'rougher.output.c

In [91]:
features_rougher_train = df_train
features_final_train = df_train


In [92]:
def func_sMAPE_one(target, predict):
    sMAPE = abs(target-predict)/((abs(target)+abs(predict))/2)*100

    #sMAPE = sMAPE.fillna(value=0)

    sMAPE = sMAPE.fillna(method='bfill')

    sMAPE = np.mean(sMAPE)

    return sMAPE


In [93]:
def func_total_sMAPE(target_rouger, pred_rougher, target_final, pred_final):

    rougher_sMAPE = func_sMAPE_one(target_rouger, pred_rougher)

    final_sMAPE = func_sMAPE_one(target_final, pred_final)

    total_sMAPE = 0.25 * rougher_sMAPE + 0.75 * final_sMAPE

    return total_sMAPE


In [94]:
def total_sMAPE(rougher_sMAPE, final_sMAPE):

    total_sMAPE = 0.25 * rougher_sMAPE + 0.75 * final_sMAPE

    return total_sMAPE


In [95]:
scorer_sMAPE = make_scorer(func_sMAPE_one, greater_is_better=False)


In [96]:
model_LinearRegression = LinearRegression()


In [97]:

scores_line_reg_rougher = cross_val_score(model_LinearRegression,
                                          features_rougher_train,
                                          target_rougher_train,
                                          scoring=scorer_sMAPE,
                                          cv=5
                                          )

rougher_sMAPE = np.mean(abs(scores_line_reg_rougher))

print('sMAPE(rougher):',
      round(rougher_sMAPE, 5))


In [98]:

scores_line_reg_rougher = cross_val_score(model_LinearRegression,
                                          features_final_train,
                                          target_final_train,
                                          scoring=scorer_sMAPE,
                                          cv=5
                                          )

final_sMAPE = np.mean(abs(scores_line_reg_rougher))

print('sMAPE(final):',
      round(final_sMAPE, 5))


In [99]:
total_sMAPE_lin_regr = total_sMAPE(rougher_sMAPE, final_sMAPE)

print('sMAPE(weighted):',
      round(total_sMAPE_lin_regr, 5))


In [100]:
%%time

model_DecisionTreeRegressor = DecisionTreeRegressor(random_state=12345)

parameters = {'max_depth': [x for x in range(1, 40)]}

grid_dt_rougher = GridSearchCV(model_DecisionTreeRegressor,
                               parameters,
                               scoring=scorer_sMAPE,
                               cv=5
                               )

grid_dt_rougher.fit(features_rougher_train, target_rougher_train)


CPU times: user 1min 25s, sys: 184 ms, total: 1min 25s
Wall time: 1min 25s


GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=12345),
             param_grid={'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                       13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
                                       23, 24, 25, 26, 27, 28, 29, 30, ...]},
             scoring=make_scorer(func_sMAPE_one, greater_is_better=False))

In [101]:
print(grid_dt_rougher.best_params_)
print()
print(grid_dt_rougher.get_params())
print()
print('sMAPE(rougher):',
      abs(round(grid_dt_rougher.best_score_, 5)))


In [102]:
%%time

model_DecisionTreeRegressor = DecisionTreeRegressor(random_state=12345)

parameters = {'max_depth': [x for x in range(1, 40)]}

grid_dt_final = GridSearchCV(model_DecisionTreeRegressor,
                             parameters,
                             scoring=scorer_sMAPE,
                             cv=5
                             )

grid_dt_final.fit(features_final_train, target_final_train)


CPU times: user 1min 33s, sys: 176 ms, total: 1min 33s
Wall time: 1min 33s


GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=12345),
             param_grid={'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                       13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
                                       23, 24, 25, 26, 27, 28, 29, 30, ...]},
             scoring=make_scorer(func_sMAPE_one, greater_is_better=False))

In [103]:
print(grid_dt_final.best_params_)
print()
print(grid_dt_final.get_params())
print()
print('sMAPE(final):',
      abs(round(grid_dt_final.best_score_, 5)))


In [104]:
total_sMAPE_tree_GridSearchCV = total_sMAPE(abs(round(grid_dt_rougher.best_score_, 5)), abs(
    round(grid_dt_final.best_score_, 5)))

print('sMAPE(weighted, tuned):',
      round(total_sMAPE_tree_GridSearchCV, 5))


In [105]:
%%time

model_RandomForestRegressor = RandomForestRegressor(random_state=12345)

parameters = {'max_depth': [x for x in range(2, 15, 3)],
              'n_estimators': [x for x in range(1, 201, 25)]
              }

grid_rf_rougher = GridSearchCV(model_RandomForestRegressor,
                               parameters,
                               scoring=scorer_sMAPE,
                               cv=5
                               )

grid_rf_rougher.fit(features_rougher_train, target_rougher_train)


CPU times: user 50min 11s, sys: 7.44 s, total: 50min 18s
Wall time: 50min 25s


GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=12345),
             param_grid={'max_depth': [2, 5, 8, 11, 14],
                         'n_estimators': [1, 26, 51, 76, 101, 126, 151, 176]},
             scoring=make_scorer(func_sMAPE_one, greater_is_better=False))

In [106]:
print(grid_rf_rougher.best_params_)
print()
print(grid_rf_rougher.get_params())
print()
print('sMAPE(rougher):',
      abs(round(grid_rf_rougher.best_score_, 5)))


In [108]:
%%time

model_RandomForestRegressor = RandomForestRegressor(random_state=12345)

parameters = {'max_depth': [x for x in range(2, 15, 3)],
              'n_estimators': [x for x in range(1, 151, 30)]
              }

grid_rf_final = GridSearchCV(model_RandomForestRegressor,
                             parameters,
                             scoring=scorer_sMAPE,
                             cv=5
                             )

grid_rf_final.fit(features_final_train, target_final_train)


CPU times: user 22min 22s, sys: 3.03 s, total: 22min 25s
Wall time: 22min 28s


GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=12345),
             param_grid={'max_depth': [2, 5, 8, 11, 14],
                         'n_estimators': [1, 31, 61, 91, 121]},
             scoring=make_scorer(func_sMAPE_one, greater_is_better=False))

In [109]:
print(grid_rf_final.best_params_)
print()
print(grid_rf_final.get_params())
print()
print('sMAPE(final):',
      abs(round(grid_rf_final.best_score_, 5)))


In [110]:
total_sMAPE_forest_GridSearchCV = total_sMAPE(abs(round(grid_rf_rougher.best_score_, 5)), abs(
    round(grid_rf_final.best_score_, 5)))

print('sMAPE(weighted, tuned):',
      round(total_sMAPE_forest_GridSearchCV, 5))


## 4. Held-out evaluation

The selected Random Forest is evaluated on the test set.


In [111]:
def RandomForestRegressor_model(features_train, target_train, features_test, depth, estimators):

    model_RandomForestRegressor = RandomForestRegressor(
        random_state=12345, max_depth=depth, n_estimators=estimators)

    model_RandomForestRegressor.fit(features_train, target_train)

    pred_test_RandomForestRegressor = model_RandomForestRegressor.predict(
        features_test)

    return pred_test_RandomForestRegressor


In [112]:
predict_rougher_test = RandomForestRegressor_model(
    features_rougher_train, target_rougher_train, features_rougher_test, 5, 151)


In [113]:
predict_final_test = RandomForestRegressor_model(
    features_final_train, target_final_train, features_final_test, 5, 121)


In [114]:
test_sMAPE = func_total_sMAPE(target_rougher_test,
                              predict_rougher_test,
                              target_final_test,
                              predict_final_test
                              )


In [115]:
print('Weighted sMAPE on the test set:', round(test_sMAPE, 5))


## 5. Baseline comparison

A constant regressor provides a direct adequacy check for the selected model.


In [116]:
def baseline_model(features_train, target_train, features_test):

    Dummy_Regressor = DummyRegressor(strategy="mean")

    Dummy_Regressor.fit(features_train, target_train)

    pred_test = Dummy_Regressor.predict(features_test)

    return pred_test


In [117]:
pred_test_rougher_const = baseline_model(
    features_rougher_train, target_rougher_train, features_rougher_test)


In [118]:
pred_test_final_const = baseline_model(
    features_final_train, target_final_train, features_final_test)


In [119]:
test_sMAPE_Dummy = func_total_sMAPE(target_rougher_test,
                                    pred_test_rougher_const,
                                    target_final_test,
                                    pred_test_final_const
                                    )


In [120]:
print('Weighted sMAPE for the constant baseline:',
      round(test_sMAPE_Dummy, 5))


## Conclusion

The Random Forest improved weighted sMAPE over the constant baseline on the held-out sample. The analysis also confirms that downstream process measurements unavailable at prediction time must be excluded to prevent leakage.
